# Drug-Drug Interaction using Deep Learning

* Upload the lab_resources and DDI_nn files to you Drive Account:
  * Lab_resource: https://www.cs.upc.edu/~turmo/mud/lab/lab_resources.zip
  * DDI_nn code: https://www.cs.upc.edu/~turmo/mud/lab/07-DDI-nn.zip
* Before running the code, ensure that your Google Colab is set to use GPU:
  * Edit → Notebook Settings
* Mount your Drive disk unit:
  * Left-side menu → Files → Mount drive (the icon that looks like a folder with the Drive logo).


Define the paths to the data and utils in your Drive unit:

In [1]:
utilsdir='drive/MyDrive/MDS/Q2/MUD/DDI-nn/07-DDI-nn'
evaluatordir='drive/MyDrive/MDS/Q2/MUD/DDI-nn/lab_resources/DDI/util'
trainfile='drive/MyDrive/MDS/Q2/MUD/DDI-nn/07-DDI-nn/train.pck'
validationfile='drive/MyDrive/MDS/Q2/MUD/DDI-nn/07-DDI-nn/devel.pck'
testfile='drive/MyDrive/MDS/Q2/MUD/DDI-nn/07-DDI-nn/test.pck'
validationdir='drive/MyDrive/MDS/Q2/MUD/DDI-nn/lab_resources/DDI/data/devel'
modelname ='model.keras'
outfile ='out.txt'
# --- IMPLEMENTATION START ---
embfile='drive/MyDrive/MDS/Q2/MUD/DDI-nn/pubmed2018_w2v_200D/pubmed2018_w2v_200D.bin'
# --- IMPLEMENTATION END ---

In [2]:
!pip install -q transformers
import sys
sys.path.insert(1,utilsdir)
sys.path.insert(1,evaluatordir)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

from contextlib import redirect_stdout

from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, Flatten, Embedding, Dense, TimeDistributed, Dropout, Bidirectional, concatenate, Softmax, GlobalMaxPooling1D, LSTM
from codemaps import *
from dataset import *

import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# --- IMPLEMENTATION START ---
BIOBERT_MODEL = "michiyasunaga/BioLinkBERT-base"
MAX_LEN_BERT = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

def get_tokenizer():
    tok = AutoTokenizer.from_pretrained(BIOBERT_MODEL)
    tok.add_special_tokens({'additional_special_tokens': ['<DRUG1>', '<DRUG2>', '<DRUG_OTHER>']})
    return tok

def encode_sentences_bert(data, tokenizer):
    texts = [' '.join(t['form'] for t in s['sent']) for s in data.sentences()]
    enc = tokenizer(texts, max_length=MAX_LEN_BERT, padding='max_length',
                    truncation=True, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask']
# --- IMPLEMENTATION END ---

Device: cuda


In [6]:
import numpy as np
from dataset import Dataset
from codemaps import Codemaps

traindata = Dataset(trainfile)
valdata = Dataset(validationfile)

max_len = 150
codes = Codemaps(traindata, max_len)

# --- IMPLEMENTATION START ---
Yt_labels = torch.tensor([codes.label2idx(s['type']) for s in traindata.sentences()], dtype=torch.long)
Yv_labels = torch.tensor([codes.label2idx(s['type']) for s in valdata.sentences()],   dtype=torch.long)

tokenizer = get_tokenizer()
X_ids_t, X_mask_t = encode_sentences_bert(traindata, tokenizer)
X_ids_v, X_mask_v = encode_sentences_bert(valdata,   tokenizer)

train_loader = DataLoader(TensorDataset(X_ids_t, X_mask_t, Yt_labels), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_ids_v, X_mask_v, Yv_labels), batch_size=32)
# --- IMPLEMENTATION END ---

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
# --- IMPLEMENTATION START ---
model = AutoModelForSequenceClassification.from_pretrained(
    BIOBERT_MODEL, num_labels=codes.get_n_labels()
)
model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=2e-5)
# --- IMPLEMENTATION END ---

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: michiyasunaga/BioLinkBERT-base
Key                          | Status     | 
-----------------------------+------------+-
bert.embeddings.position_ids | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [8]:
## --------- MAIN PROGRAM -----------
from tqdm.notebook import tqdm
from transformers import get_linear_schedule_with_warmup

def eval_metrics(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            total_loss += out.loss.item() * labels.size(0)
            correct += (out.logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)
    return correct / total, total_loss / total

# --- IMPLEMENTATION START ---
NUM_EPOCHS = 10
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps)

best_val_acc = 0.0
patience_count = 0
PATIENCE = 3

for epoch in range(NUM_EPOCHS):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS}", unit="step")

    for ids, mask, labels in pbar:
        ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        run_loss += out.loss.item() * labels.size(0)
        correct  += (out.logits.argmax(dim=-1) == labels).sum().item()
        total    += labels.size(0)
        pbar.set_postfix(loss=f"{run_loss/total:.4f}", acc=f"{correct/total:.4f}")

    val_acc, val_loss = eval_metrics(model, val_loader)
    print(f"  → val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'biobert_best.pt')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(torch.load('biobert_best.pt'))
codes.save(modelname)
# --- IMPLEMENTATION END ---

Epoch 01/10:   0%|          | 0/1447 [00:00<?, ?step/s]

  → val_loss=0.4152  val_acc=0.8442


Epoch 02/10:   0%|          | 0/1447 [00:00<?, ?step/s]

  → val_loss=0.2111  val_acc=0.9450


Epoch 03/10:   0%|          | 0/1447 [00:00<?, ?step/s]

...

  → val_loss=0.2821  val_acc=0.9604


Epoch 09/10:   0%|          | 0/1447 [00:00<?, ?step/s]

  → val_loss=0.2985  val_acc=0.9604


Epoch 10/10:   0%|          | 0/1447 [00:00<?, ?step/s]

  → val_loss=0.2942  val_acc=0.9601


# Predict

In [9]:
#import sys
import evaluator

In [10]:
def output_interactions(data, preds, outfile) :

   #print(testdata[0])
   outf = open(outfile, 'w')
   for exmp,tag in zip(data.sentences(),preds) :
      sid = exmp['sid']
      e1 = exmp['e1']
      e2 = exmp['e2']
      if tag!='null' :
         print(sid, e1, e2, tag, sep="|", file=outf)

   outf.close()

In [11]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("DDI", datadir, outfile)


In [13]:
# --- IMPLEMENTATION START ---
model.eval()
all_preds = []
X_ids_pred, X_mask_pred = encode_sentences_bert(valdata, tokenizer)
pred_loader = DataLoader(TensorDataset(X_ids_pred, X_mask_pred), batch_size=32)

with torch.no_grad():
    for ids, mask in pred_loader:
        ids, mask = ids.to(DEVICE), mask.to(DEVICE)
        logits = model(input_ids=ids, attention_mask=mask).logits
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())

Y = [codes.idx2label(int(p)) for p in all_preds]
# --- IMPLEMENTATION END ---

output_interactions(valdata, Y, outfile)
evaluation(validationdir, outfile)

                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
advise            129	  34	  12	 163	 141	79.1%	91.5%	84.9%
effect            255	  46	  57	 301	 312	84.7%	81.7%	83.2%
int                25	   5	   3	  30	  28	83.3%	89.3%	86.2%
mechanism         236	  30	  25	 266	 261	88.7%	90.4%	89.6%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	84.0%	88.2%	86.0%
------------------------------------------------------------------------------
m.avg             645	 115	  97	 760	 742	84.9%	86.9%	85.9%
m.avg(no class)   674	  86	  68	 760	 742	88.7%	90.8%	89.7%
